In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

print("="*80)
print("COMBINED ML + STRATEGY BACKTESTING")
print("="*80)
print(f"Testing Period: {TEST_START} to {TEST_END}")
print("="*80)


COMBINED ML + STRATEGY BACKTESTING
Testing Period: 2024-01-01 to 2024-12-31


In [2]:
def add_scalping_signals(data):
    df = data.copy()

    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))

    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    ema_12 = df["Close"].ewm(span=12).mean()
    ema_26 = df["Close"].ewm(span=26).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9).mean()
    macd_hist = macd - macd_signal

    buy_uptrend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = (rsi > 30) & (rsi < 50)
    buy_macd = (macd > 0) & (macd_hist > 0)

    close_20_high = df["Close"].rolling(20).max()
    buy_strength = df["Close"] > 0.95 * close_20_high

    buy_signal = (
        (buy_uptrend & buy_rsi) |
        (buy_uptrend & buy_macd) |
        (buy_uptrend & buy_strength)
    )

    sell_downtrend = (df["Close"] < sma_20) & (sma_20 < sma_50)
    sell_rsi = (rsi < 70) & (rsi > 50)
    sell_macd = (macd < 0) & (macd_hist < 0)

    sell_signal = (
        (sell_downtrend & sell_rsi) |
        (sell_downtrend & sell_macd)
    )

    signal = pd.Series(0, index=df.index)
    signal[buy_signal & ~sell_signal] = 1
    signal[sell_signal & ~buy_signal] = -1
    signal[~buy_signal & ~sell_signal] = 0

    df["strategy_signal"] = signal
    return df


def add_basic_features(data, horizon=3, cost=0.0003):
    df = data.copy()

    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()

    df["trend_10"] = (df["Close"] - sma_10) / (sma_10 + 1e-8)
    df["trend_20"] = (df["Close"] - sma_20) / (sma_20 + 1e-8)
    df["trend_diff"] = (sma_10 - sma_20) / (sma_20 + 1e-8)

    df["range_pct"] = (df["High"] - df["Low"]) / (df["Close"] + 1e-8)
    df["body_pct"] = (df["Close"] - df["Open"]) / (df["Close"] + 1e-8)
    df["body_abs"] = df["body_pct"].abs()

    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / (df["volatility_10"].rolling(50).mean() + 1e-8)
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    if "Volume" in df.columns and float(df["Volume"].sum()) > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    future_return = (df["Close"].shift(-horizon) - df["Close"]) / df["Close"]
    df["target"] = (future_return > cost).astype(int)

    df.dropna(inplace=True)
    return df


In [3]:
ticker = DEFAULT_TICKERS[0]
print(f"\n{'='*80}")
print(f"ANALYZING {ticker}")
print(f"{'='*80}")

raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)
train_data, test_data = split_data_by_date(cleaned_data)

print(f"Train data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

train_with_signals = add_scalping_signals(train_data)
test_with_signals  = add_scalping_signals(test_data)

train_with_features = add_basic_features(train_with_signals)
test_with_features  = add_basic_features(test_with_signals)

print(f"Train with features: {train_with_features.shape}")
print(f"Test with features:  {test_with_features.shape}")



ANALYZING NIFTY BANK
2026-01-31 13:23:52 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2026-01-31 13:23:53 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2026-01-31 13:23:53 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2026-01-31 13:23:53 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2026-01-31 13:23:53 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2026-01-31 13:23:53 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2026-01-31 13:23:53 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2026-01-31 13:23:53 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 0

In [4]:
INITIAL_CAPITAL = 1_000_000
RISK_FREE_RATE = 0.0

HORIZON = 9

ENTRY_Q = 0.35
ENTRY_THRESHOLD_MIN = 0.35

MAX_POSITION = 0.96
MIN_POSITION = 0.25
SIZE_EXPONENT = 0.22

BASE_STOP_LOSS = 0.024
BASE_TAKE_PROFIT = 0.032
TRAIL_STOP = 0.0025

COST_PER_TRADE = 0.000001

COOLDOWN = 0
MIN_TRADES_PER_DAY = 1

REQUIRE_UPTREND = False
TREND_STRENGTH_MIN = 0.0

MAX_VOLATILITY = 2.0

print("=" * 70)

print("=" * 70)
print(f"Configuration:")
print(f"   ENTRY_Q: {ENTRY_Q*100:.0f}% ")
print(f"   MIN_POSITION: {MIN_POSITION*100:.0f}% ")
print(f"   TAKE_PROFIT: {BASE_TAKE_PROFIT*100:.1f}%")
print(f"   TRAIL_STOP: {TRAIL_STOP*100:.1f}%")
print(f"   Ensemble Model + 25 Advanced Features")
print(f"   Dynamic ATR-based stops")
print(f"   Adaptive position sizing")
print("=" * 70)


Configuration:
   ENTRY_Q: 35% 
   MIN_POSITION: 25% 
   TAKE_PROFIT: 3.2%
   TRAIL_STOP: 0.2%
   Ensemble Model + 25 Advanced Features
   Dynamic ATR-based stops
   Adaptive position sizing


In [5]:
def add_advanced_features(df):
    d = df.copy()
    
    d['momentum_5'] = (d['Close'] - d['Close'].shift(5)) / d['Close'].shift(5)
    d['momentum_10'] = (d['Close'] - d['Close'].shift(10)) / d['Close'].shift(10)
    d['momentum_20'] = (d['Close'] - d['Close'].shift(20)) / d['Close'].shift(20)
    
    d['momentum_accel'] = d['momentum_5'] - d['momentum_10']
    
    d['tr'] = np.maximum(
        d['High'] - d['Low'],
        np.maximum(
            abs(d['High'] - d['Close'].shift(1)),
            abs(d['Low'] - d['Close'].shift(1))
        )
    )
    d['atr'] = d['tr'].rolling(14).mean()
    d['atr_pct'] = d['atr'] / d['Close']
    
    sma_10 = d['Close'].rolling(10).mean()
    sma_20 = d['Close'].rolling(20).mean()
    sma_50 = d['Close'].rolling(50).mean()
    
    d['trend_strength'] = (
        ((d['Close'] > sma_20).astype(int) * 0.4) +
        ((sma_20 > sma_50).astype(int) * 0.3) +
        ((d['momentum_5'] > 0).astype(int) * 0.3)
    )
    
    d['vol_20'] = d['Close'].pct_change().rolling(20).std()
    d['vol_regime'] = (d['vol_20'] > d['vol_20'].rolling(50).mean()).astype(int)
    
    d['high_20'] = d['High'].rolling(20).max()
    d['low_20'] = d['Low'].rolling(20).min()
    d['price_position'] = (d['Close'] - d['low_20']) / (d['high_20'] - d['low_20'] + 1e-8)
    
    d['momentum_trend_signal'] = d['momentum_5'] * d['trend_strength']
    
    d['price_momentum_align'] = d['price_position'] * d['momentum_10']
    
    return d


In [6]:
ticker = "NIFTY BANK"
print(f"\nBacktesting: {ticker}")

raw = load_kaggle_data(ticker)
cleaned = clean_ohlcv_data(raw)
train_data, test_data = split_data_by_date(cleaned)

full_data = pd.concat([train_data, test_data], axis=0)

full_with_signals = add_scalping_signals(full_data)
full_features = add_basic_features(full_with_signals)
full_features = add_advanced_features(full_features)

test_df = full_features.loc[test_data.index]

feature_cols = [
    c for c in test_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume", "strategy_signal", "tr", "high_20", "low_20"]
]

X_test = test_df[feature_cols]
y_test = test_df["target"]
prices = test_df["Close"].values
atr_values = test_df["atr"].values

print(f"Advanced features added: {len(feature_cols)} features total")



Backtesting: NIFTY BANK
2026-01-31 13:23:53 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2026-01-31 13:23:54 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2026-01-31 13:23:54 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2026-01-31 13:23:54 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2026-01-31 13:23:55 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2026-01-31 13:23:55 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2026-01-31 13:23:55 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2026-01-31 13:23:55 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-1

In [7]:
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

train_with_signals = add_scalping_signals(train_data)
train_df = add_basic_features(train_with_signals)
train_df = add_advanced_features(train_df)

feature_cols = [
    c for c in train_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume", "strategy_signal", "tr", "high_20", "low_20"]
]

X_train = train_df[feature_cols]
y_train = train_df["target"]

X_test = test_df[feature_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n Training Ensemble Model (XGBoost + LightGBM)...")

xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.025,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=8,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=0.5,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train_scaled, y_train)
xgb_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

lgb_model = LGBMClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.75,
    colsample_bytree=0.75,
    num_leaves=31,
    min_child_samples=5,
    reg_alpha=0.1,
    reg_lambda=0.5,
    objective="binary",
    metric="auc",
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train_scaled, y_train)
lgb_proba = lgb_model.predict_proba(X_test_scaled)[:, 1]

ml_prob = (xgb_proba * 0.55 + lgb_proba * 0.45)

print(f" Ensemble Model Ready")
print(f"   XGBoost AUC: {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"   LightGBM AUC: {roc_auc_score(y_test, lgb_proba):.4f}")
print(f"   Ensemble AUC: {roc_auc_score(y_test, ml_prob):.4f}")



 Training Ensemble Model (XGBoost + LightGBM)...
 Ensemble Model Ready
   XGBoost AUC: 0.6052
   LightGBM AUC: 0.6052
   Ensemble AUC: 0.6053


In [8]:
def position_size(prob, threshold):
    size = (prob - threshold) / (1 - threshold)
    return np.clip(size, 0, 1)

print(f"\n ML_PROB Ready for Backtest:")
print(f"   Shape: {ml_prob.shape}")
print(f"   Min: {ml_prob.min():.4f}, Max: {ml_prob.max():.4f}, Mean: {ml_prob.mean():.4f}")



 ML_PROB Ready for Backtest:
   Shape: (80907,)
   Min: 0.1085, Max: 0.5387, Mean: 0.2942


In [9]:
capital = INITIAL_CAPITAL
equity_curve = []
trades = []
recent_returns = []

print(f"\n ML Ensemble Probability Distribution:")
print(f"  Min: {ml_prob.min():.4f}")
print(f"  25%: {np.percentile(ml_prob, 25):.4f}")
print(f"  Median: {np.median(ml_prob):.4f}")
print(f"  75%: {np.percentile(ml_prob, 75):.4f}")
print(f"  95%: {np.percentile(ml_prob, 95):.4f}")
print(f"  Max: {ml_prob.max():.4f}")

ENTRY_THRESHOLD = np.quantile(ml_prob, ENTRY_Q)
print(f"\n Entry Threshold (Q={ENTRY_Q}): {ENTRY_THRESHOLD:.4f}")
print(f" Potential signals: {(ml_prob >= ENTRY_THRESHOLD).sum()} / {len(ml_prob)}")

i = 0
n = len(prices)
last_exit = -COOLDOWN

while i < n - HORIZON:

    prob = ml_prob[i]
    current_price = prices[i]
    current_atr = atr_values[i]
    
    if i < 50:
        equity_curve.append(capital)
        i += 1
        continue
    
    if i - last_exit < COOLDOWN:
        equity_curve.append(capital)
        i += 1
        continue
    
    if prob < ENTRY_THRESHOLD:
        equity_curve.append(capital)
        i += 1
        continue
    
    if REQUIRE_UPTREND:
        trend_strength = test_df["trend_strength"].iloc[i]
        if trend_strength < 0.6:
            equity_curve.append(capital)
            i += 1
            continue
    
    momentum_5 = test_df["momentum_5"].iloc[i]
    if momentum_5 < TREND_STRENGTH_MIN:
        equity_curve.append(capital)
        i += 1
        continue
    
    vol_regime = test_df["vol_regime"].iloc[i]
    if vol_regime == 1 and prob < 0.55:
        equity_curve.append(capital)
        i += 1
        continue
    
    if recent_returns:
        recent_wins = sum(1 for r in recent_returns[-20:] if r > 0)
        recent_win_rate = recent_wins / min(20, len(recent_returns[-20:]))
        size_multiplier = 0.8 + (recent_win_rate * 0.4)
    else:
        size_multiplier = 1.0
    
    confidence_edge = prob - ENTRY_THRESHOLD
    max_edge = 1.0 - ENTRY_THRESHOLD
    normalized = confidence_edge / max_edge if max_edge > 0 else 0.5
    
    position_fraction = MIN_POSITION + (MAX_POSITION - MIN_POSITION) * (normalized ** SIZE_EXPONENT)
    position_fraction = np.clip(position_fraction * size_multiplier, MIN_POSITION, MAX_POSITION)
    
    position_value = capital * position_fraction
    entry_price = current_price
    max_price = current_price
    
    vol_adjusted_stop = BASE_STOP_LOSS + (current_atr / current_price) * 0.5
    vol_adjusted_stop = np.clip(vol_adjusted_stop, BASE_STOP_LOSS, BASE_STOP_LOSS * 1.5)
    
    vol_adjusted_profit = BASE_TAKE_PROFIT * (0.8 + vol_regime * 0.4)
    
    exit_price = prices[i + HORIZON]
    exit_idx = i + HORIZON
    exit_reason = "TIME"
    
    for j in range(1, HORIZON + 1):
        price = prices[i + j]
        max_price = max(max_price, price)
        
        if price <= entry_price * (1 - vol_adjusted_stop):
            exit_price = entry_price * (1 - vol_adjusted_stop)
            exit_idx = i + j
            exit_reason = "STOP"
            break
        
        if price >= entry_price * (1 + vol_adjusted_profit):
            exit_price = entry_price * (1 + vol_adjusted_profit)
            exit_idx = i + j
            exit_reason = "PROFIT"
            break
        
        trail_level = max_price * (1 - TRAIL_STOP)
        if price < trail_level:
            exit_price = trail_level
            exit_idx = i + j
            exit_reason = "TRAIL"
            break
    
    ret = (exit_price - entry_price) / entry_price
    net_ret = ret - (COST_PER_TRADE * 2)
    
    pnl = position_value * net_ret
    capital += pnl
    
    if capital < 0:
        capital = INITIAL_CAPITAL * 0.001
    
    recent_returns.append(net_ret)
    
    trades.append({
        "entry_idx": i,
        "exit_idx": exit_idx,
        "entry_price": entry_price,
        "exit_price": exit_price,
        "prob": prob,
        "size": position_fraction,
        "return": net_ret,
        "pnl": pnl,
        "capital": capital,
        "exit_reason": exit_reason,
        "vol": test_df["volatility_10"].iloc[i],
        "trend_strength": test_df["trend_strength"].iloc[i]
    })
    
    equity_curve.append(capital)
    last_exit = exit_idx
    i = exit_idx + COOLDOWN

print(f"\n Advanced Backtest complete: {len(trades)} trades")
print(f" Capital: {INITIAL_CAPITAL:,.0f} → {capital:,.0f}")



 ML Ensemble Probability Distribution:
  Min: 0.1085
  25%: 0.2436
  Median: 0.2966
  75%: 0.3439
  95%: 0.4128
  Max: 0.5387

 Entry Threshold (Q=0.35): 0.2654
 Potential signals: 52589 / 80907

 Advanced Backtest complete: 2914 trades
 Capital: 1,000,000 → 1,151,022


In [10]:


from xgboost import XGBClassifier

feature_cols = [
    col for col in train_df.columns
    if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume', 'strategy_signal']
]

X_train_bt = train_df[feature_cols]
y_train_bt = train_df['target']
X_test_bt = test_df[feature_cols]

from sklearn.preprocessing import StandardScaler
scaler_bt = StandardScaler()
X_train_scaled_bt = scaler_bt.fit_transform(X_train_bt)
X_test_scaled_bt = scaler_bt.transform(X_test_bt)

scale_pos_weight_bt = (len(X_train_scaled_bt) - y_train_bt.sum()) / max(y_train_bt.sum(), 1)

model_bt = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.02,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=10,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=0.5,
    scale_pos_weight=scale_pos_weight_bt,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

model_bt.fit(X_train_scaled_bt, y_train_bt)

ml_prob = model_bt.predict_proba(X_test_scaled_bt)[:, 1]





In [11]:
equity = pd.Series(equity_curve)
returns = equity.pct_change().dropna()

total_return = (equity.iloc[-1] / equity.iloc[0]) - 1
max_dd = ((equity / equity.cummax()) - 1).min()

sharpe = (
    returns.mean() / returns.std()
    if returns.std() > 0 else 0
) * np.sqrt(252 * 6.5 * 60)

winning_trades = [t["pnl"] for t in trades if t["pnl"] > 0]
losing_trades = [t["pnl"] for t in trades if t["pnl"] < 0]

gross_profit = sum(winning_trades) if winning_trades else 0
gross_loss = abs(sum(losing_trades)) if losing_trades else 1e-8

profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf

win_rate = len(winning_trades) / len(trades) if trades else 0

avg_win = np.mean(winning_trades) if winning_trades else 0
avg_loss = np.mean(losing_trades) if losing_trades else 0
expectancy = (win_rate * avg_win) + ((1 - win_rate) * avg_loss)

print("\n" + "="*60)
print(" BACKTEST RESULTS")
print("="*60)
print(f"Initial Capital:   ₹{equity.iloc[0]:,.0f}")
print(f"Final Capital:     ₹{equity.iloc[-1]:,.0f}")
print(f"Total Return:      {total_return*100:.2f}%")
print(f"Net Profit:        ₹{equity.iloc[-1] - equity.iloc[0]:,.0f}")
print(f"Max Drawdown:      {max_dd*100:.2f}%")
print(f"Sharpe Ratio:      {sharpe:.2f}")
print(f"Profit Factor:     {profit_factor:.2f}")
print(f"Win Rate:          {win_rate*100:.2f}%")
print(f"Avg Win:           ₹{avg_win:,.0f}")
print(f"Avg Loss:          ₹{avg_loss:,.0f}")
print(f"Total Trades:      {len(trades)}")

print("="*60)

if len(trades) < 100:
    print(f" WARNING: Only {len(trades)} trades - sample size too small")
    print("  Need 200+ trades for weak edge (AUC 0.60) to show in backtest")







 BACKTEST RESULTS
Initial Capital:   ₹1,000,000
Final Capital:     ₹1,151,022
Total Return:      15.10%
Net Profit:        ₹151,022
Max Drawdown:      -2.63%
Sharpe Ratio:      3.51
Profit Factor:     1.17
Win Rate:          49.18%
Avg Win:           ₹742
Avg Loss:          ₹-616
Total Trades:      2914


In [12]:
import yfinance as yf
from datetime import datetime, timedelta
import pytz

IST = pytz.timezone("Asia/Kolkata")

print("\n" + "="*80)
print(" LIVE PAPER TRADING - LAST 7 DAYS (YFINANCE)")
print("="*80)

print("\n Fetching last 7 days of NIFTY BANK data from yfinance...")

try:
    end_date = datetime.now(IST)
    start_date = end_date - timedelta(days=7)
    
    try:
        live_data = yf.download("^NSEBANK", start=start_date, end=end_date, interval="1m", progress=False)
        ticker_name = "^NSEBANK"
    except:
        live_data = yf.download("NIFTYBANK.NS", start=start_date, end=end_date, interval="1m", progress=False)
        ticker_name = "NIFTYBANK.NS"
    # --- FORCE CONVERT YFINANCE TIMESTAMPS TO IST ---
    if live_data.index.tz is None:
        live_data.index = live_data.index.tz_localize("UTC").tz_convert(IST)
    else:
        live_data.index = live_data.index.tz_convert(IST)
        
    
    if isinstance(live_data.columns, pd.MultiIndex):
        live_data.columns = [col[0] for col in live_data.columns]
    
    live_data.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    live_data = live_data[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
    
    print(f" Fetched {len(live_data)} candles from {live_data.index[0]} to {live_data.index[-1]}")
    print(f"   Ticker: {ticker_name}")
    
except Exception as e:
    print(f" Error fetching data: {e}")
    print("   Using last 7 days from test_df as fallback...")
    live_data = test_df[['Open', 'High', 'Low', 'Close', 'Volume']].iloc[-len(test_df)//50:]
    print(f" Using {len(live_data)} candles from test data")

print("\n Applying feature engineering to live data...")

try:
    live_with_signals = add_scalping_signals(live_data.copy())
    live_with_features = add_basic_features(live_with_signals.copy())
    live_features = add_advanced_features(live_with_features.copy())
    live_features = live_features.dropna()
    
    feature_cols_live = [
        c for c in live_features.columns
        if c not in ["target", "Open", "High", "Low", "Close", "Volume", "strategy_signal"]
    ]
    
    X_live = live_features[feature_cols_live].copy()
    X_live = X_live[feature_cols]
    
    X_live_scaled = scaler_bt.transform(X_live)
    ml_prob_live = model_bt.predict_proba(X_live_scaled)[:, 1]
    
    live_prices = live_features["Close"].values
    live_atr = live_features["atr"].values
    
    print(f" Features applied: {len(feature_cols_live)} features")
    print(f"   ML Prob range: {ml_prob_live.min():.4f} to {ml_prob_live.max():.4f}")
    print(f"   Data points: {len(live_features)} candles")
    
except Exception as e:
    print(f"  Error in feature pipeline: {e}")
    print("   Attempting fallback approach...")
    
    live_features = test_df.iloc[-len(test_df)//50:].copy()
    
    try:
        X_live = live_features[feature_cols].copy()
        X_live_scaled = scaler_bt.transform(X_live)
        ml_prob_live = model_bt.predict_proba(X_live_scaled)[:, 1]
        
        live_prices = live_features["Close"].values
        live_atr = live_features["atr"].values
        
        print(f" Fallback successful! Using {len(live_features)} candles from test data")
        
    except Exception as e2:
        print(f" Fallback also failed: {e2}")
        raise

TODAY_ENTRY_THRESHOLD_CALC = np.percentile(ml_prob_live, 75)
TODAY_STOP_LOSS = 0.0150
TODAY_TAKE_PROFIT_1 = 0.0040
TODAY_TAKE_PROFIT_2 = 0.0080
TODAY_TAKE_PROFIT_3 = 0.0150
TODAY_TRAIL_STOP = 0.0030
TODAY_HORIZON = 20
TODAY_MIN_POSITION = 0.15
TODAY_MAX_POSITION = 0.50
TODAY_COOLDOWN = 0

LIVE_ENTRY_THRESHOLD = np.percentile(ml_prob_live, 75)
LIVE_STOP_LOSS = TODAY_STOP_LOSS
LIVE_TAKE_PROFIT_1 = TODAY_TAKE_PROFIT_1
LIVE_TAKE_PROFIT_2 = TODAY_TAKE_PROFIT_2
LIVE_TAKE_PROFIT_3 = TODAY_TAKE_PROFIT_3
LIVE_TRAIL_STOP = TODAY_TRAIL_STOP
LIVE_HORIZON = TODAY_HORIZON
LIVE_MIN_POSITION = TODAY_MIN_POSITION
LIVE_MAX_POSITION = TODAY_MAX_POSITION

print(f"\n LIVE PAPER TRADING CONFIGURATION (7 DAYS - HYBRID OPTIMAL v2):")
print(f"   Entry Threshold: {LIVE_ENTRY_THRESHOLD:.4f}")
print(f"   Stop Loss: {LIVE_STOP_LOSS*100:.2f}%")
print(f"   1st Target: {LIVE_TAKE_PROFIT_1*100:.2f}%")
print(f"   2nd Target: {LIVE_TAKE_PROFIT_2*100:.2f}%")
print(f"   3rd Target: {LIVE_TAKE_PROFIT_3*100:.2f}%")
print(f"   Trailing Stop: {LIVE_TRAIL_STOP*100:.2f}%")
print(f"   Holding Period: {LIVE_HORIZON} bars")
print(f"   Position Size: {LIVE_MIN_POSITION*100:.0f}% - {LIVE_MAX_POSITION*100:.0f}%")
print("="*80)
print(f"\n Starting Live Paper Trading ({len(live_features)} candles)...\n")

live_paper_capital = INITIAL_CAPITAL
live_paper_peak_capital = live_paper_capital
live_paper_positions = []
live_paper_trades = []
live_paper_equity_curve = []
live_paper_last_exit_idx = -TODAY_COOLDOWN

for i in range(len(live_features)):
    current_price = live_prices[i]
    current_time = live_features.index[i]
    current_ml_prob = ml_prob_live[i]
    current_atr = live_atr[i]
    
    if i < 30:
        live_paper_equity_curve.append(live_paper_capital)
        continue
    
    positions_to_close = []
    
    for pos_idx, pos in enumerate(live_paper_positions):
        price_move = (current_price - pos['entry_price']) / pos['entry_price']
        
        vol_adjusted_stop = LIVE_STOP_LOSS * (1 + (current_atr / current_price) * 0.5)
        vol_adjusted_stop = np.clip(vol_adjusted_stop, LIVE_STOP_LOSS * 0.8, LIVE_STOP_LOSS * 1.2)
        
        exit_hit = False
        exit_reason = None
        exit_price = current_price
        
        if price_move >= LIVE_TAKE_PROFIT_1:
            exit_hit = True
            exit_reason = "QUICK_PROFIT"
            exit_price = pos['entry_price'] * (1 + LIVE_TAKE_PROFIT_1)
        
        elif price_move >= LIVE_TAKE_PROFIT_2:
            exit_hit = True
            exit_reason = "MEDIUM_PROFIT"
            exit_price = pos['entry_price'] * (1 + LIVE_TAKE_PROFIT_2)
        
        elif price_move >= LIVE_TAKE_PROFIT_3:
            exit_hit = True
            exit_reason = "BIG_PROFIT"
            exit_price = pos['entry_price'] * (1 + LIVE_TAKE_PROFIT_3)
        
        elif price_move > LIVE_TAKE_PROFIT_1 and current_price < pos['max_price'] * (1 - LIVE_TRAIL_STOP):
            exit_hit = True
            exit_reason = "TRAIL"
            exit_price = pos['max_price'] * (1 - LIVE_TRAIL_STOP)
        
        elif price_move <= -vol_adjusted_stop:
            exit_hit = True
            exit_reason = "STOP"
            exit_price = current_price
        
        elif i - pos['entry_idx'] >= LIVE_HORIZON:
            exit_hit = True
            exit_reason = "TIME"
            exit_price = current_price
        
        if exit_hit:
            ret = (exit_price - pos['entry_price']) / pos['entry_price']
            net_ret = ret - (COST_PER_TRADE * 2)
            pnl = pos['position_value'] * net_ret
            
            live_paper_capital += pnl
            live_paper_peak_capital = max(live_paper_peak_capital, live_paper_capital)
            
            live_paper_trades.append({
                'entry_idx': pos['entry_idx'],
                'exit_idx': i,
                'entry_price': pos['entry_price'],
                'exit_price': exit_price,
                'entry_time': pos['entry_time'],
                'exit_time': current_time,
                'prob': pos['ml_prob'],
                'size': pos['position_fraction'],
                'return': net_ret,
                'pnl': pnl,
                'capital_after': live_paper_capital,
                'exit_reason': exit_reason
            })
            
            emoji = "🎯" if pnl > 0 else "🛑"
            pnl_str = f"+₹{pnl:,.0f}" if pnl > 0 else f"-₹{abs(pnl):,.0f}"
            ret_str = f"+{net_ret*100:.2f}%" if net_ret > 0 else f"{net_ret*100:.2f}%"
            
            print(f"{emoji} EXIT | {current_time.strftime('%Y-%m-%d %H:%M:%S')} | ₹{exit_price:.2f} | P&L: {pnl_str} ({ret_str}) | [{exit_reason}] | Capital: ₹{live_paper_capital:,.0f}")
            
            positions_to_close.append(pos_idx)
            live_paper_last_exit_idx = i
    
    live_paper_positions = [pos for idx, pos in enumerate(live_paper_positions) if idx not in positions_to_close]
    
    if current_ml_prob >= LIVE_ENTRY_THRESHOLD and i < len(live_features) - LIVE_HORIZON:
        momentum_5 = live_features["momentum_5"].iloc[i]
        if momentum_5 >= TREND_STRENGTH_MIN * 0.5:  
            
            total_exposure = sum(p['position_fraction'] for p in live_paper_positions)
            max_allowed = 1.5
            if total_exposure < max_allowed:
                
                confidence_edge = current_ml_prob - LIVE_ENTRY_THRESHOLD
                max_edge = 1.0 - LIVE_ENTRY_THRESHOLD
                normalized = confidence_edge / max_edge if max_edge > 0 else 0.4
                
                position_fraction = LIVE_MIN_POSITION + (LIVE_MAX_POSITION - LIVE_MIN_POSITION) * (normalized ** 1.3)
                position_fraction = np.clip(position_fraction, LIVE_MIN_POSITION, LIVE_MAX_POSITION)
                
                position_fraction *= (1.0 - total_exposure / max_allowed * 0.3)
                position_fraction = max(position_fraction, LIVE_MIN_POSITION * 0.5)
                
                position_value = live_paper_capital * position_fraction
                
                live_paper_positions.append({
                    'entry_idx': i,
                    'entry_price': current_price,
                    'entry_time': current_time,
                    'position_value': position_value,
                    'position_fraction': position_fraction,
                    'ml_prob': current_ml_prob,
                    'max_price': current_price
                })
                
                prob_pct = current_ml_prob * 100
                size_pct = position_fraction * 100
                print(f"🟢 ENTRY #{len(live_paper_positions)} | {current_time.strftime('%Y-%m-%d %H:%M:%S')} | ₹{current_price:.2f} | Conf: {prob_pct:.1f}% | Size: {size_pct:.0f}% | Capital: ₹{live_paper_capital:,.0f}")
    
    for pos in live_paper_positions:
        pos['max_price'] = max(pos['max_price'], current_price)
    
    live_paper_equity_curve.append(live_paper_capital)

if live_paper_positions:
    print(f"\n⏹️  Market Close - Closing {len(live_paper_positions)} open position(s)...")
    for pos in live_paper_positions:
        exit_price = live_prices[-1]
        ret = (exit_price - pos['entry_price']) / pos['entry_price']
        net_ret = ret - (COST_PER_TRADE * 2)
        pnl = pos['position_value'] * net_ret
        
        live_paper_capital += pnl
        live_paper_peak_capital = max(live_paper_peak_capital, live_paper_capital)
        
        live_paper_trades.append({
            'entry_idx': pos['entry_idx'],
            'exit_idx': len(live_features) - 1,
            'entry_price': pos['entry_price'],
            'exit_price': exit_price,
            'entry_time': pos['entry_time'],
            'exit_time': live_features.index[-1],
            'prob': pos['ml_prob'],
            'size': pos['position_fraction'],
            'return': net_ret,
            'pnl': pnl,
            'capital_after': live_paper_capital,
            'exit_reason': 'CLOSE'
        })
        
        print(f"🛑 CLOSE | Final Price: ₹{exit_price:.2f} | P&L: ₹{pnl:,.0f} ({net_ret*100:+.2f}%)")

print("\n" + "="*80)
print("📊 LIVE PAPER TRADING RESULTS (LAST 7 DAYS)")
print("="*80)

live_paper_equity = np.array(live_paper_equity_curve)
live_paper_total_return = (live_paper_equity[-1] / live_paper_equity[0]) - 1
live_paper_total_pnl = live_paper_equity[-1] - live_paper_equity[0]
live_paper_max_dd = ((live_paper_equity / np.maximum.accumulate(live_paper_equity)) - 1).min()

live_paper_returns = np.diff(live_paper_equity) / live_paper_equity[:-1]
live_paper_returns = live_paper_returns[live_paper_returns != 0]
if len(live_paper_returns) > 0 and live_paper_returns.std() > 0:
    live_paper_sharpe = (live_paper_returns.mean() / live_paper_returns.std()) * np.sqrt(252 * 6.5 * 60)
else:
    live_paper_sharpe = 0

if live_paper_trades:
    live_winning = sum(1 for t in live_paper_trades if t['pnl'] > 0)
    live_losing = len(live_paper_trades) - live_winning
    live_paper_win_rate = live_winning / len(live_paper_trades)
    
    live_gross_profit = sum(max(0, t['pnl']) for t in live_paper_trades)
    live_gross_loss = abs(sum(min(0, t['pnl']) for t in live_paper_trades))
    live_paper_pf = live_gross_profit / live_gross_loss if live_gross_loss > 0 else np.inf
    
    live_avg_win = live_gross_profit / live_winning if live_winning > 0 else 0
    live_avg_loss = live_gross_loss / live_losing if live_losing > 0 else 0
else:
    live_winning = live_losing = live_paper_win_rate = live_paper_pf = live_avg_win = live_avg_loss = 0

print(f"\n Capital:")
print(f"   Initial: ₹{live_paper_equity[0]:,.0f}")
print(f"   Final: ₹{live_paper_equity[-1]:,.0f}")
print(f"   Peak: ₹{live_paper_peak_capital:,.0f}")
print(f"   Net P&L: ₹{live_paper_total_pnl:,.0f}")

print(f"\n Performance:")
print(f"   Total Return: {live_paper_total_return*100:+.2f}%")
print(f"   Max Drawdown: {live_paper_max_dd*100:.2f}%")
print(f"   Sharpe Ratio: {live_paper_sharpe:.2f}")
print(f"   Profit Factor: {live_paper_pf:.2f}")

print(f"\n Trading Statistics:")
print(f"   Total Trades: {len(live_paper_trades)}")
print(f"   Winning: {live_winning} ({live_paper_win_rate*100:.1f}%)")
print(f"   Losing: {live_losing} ({(1-live_paper_win_rate)*100:.1f}%)")
print(f"   Avg Win: ₹{live_avg_win:,.0f}")
print(f"   Avg Loss: ₹{live_avg_loss:,.0f}")
if live_avg_loss != 0:
    print(f"   Win/Loss Ratio: {live_avg_win / abs(live_avg_loss):.2f}x")

print("\n" + "="*80)

print(f"\n COMPARISON (BACKTEST vs OFFLINE vs LIVE):")
print(f"   Backtest (2024):        {total_return*100:+.2f}% on {len(trades)} trades")
print(f"   Live Paper (7 days):    {live_paper_total_return*100:+.2f}% on {len(live_paper_trades)} trades")

print(f"\n{'='*80}")
if live_paper_total_return > 0.10:
    print(f" EXCELLENT! +{live_paper_total_return*100:.2f}% return in just 7 days!")
elif live_paper_total_return > 0.05:
    print(f" VERY GOOD! +{live_paper_total_return*100:.2f}% return in 7 days")
elif live_paper_total_return > 0:
    print(f" PROFITABLE! +{live_paper_total_return*100:.2f}% return in 7 days")
elif live_paper_total_return < 0:
    print(f"  LIMITED DATA. {live_paper_total_return*100:.2f}% on {len(live_paper_trades)} trades in 7 days")
    if len(live_paper_trades) < 10:
        print("   → Need more days of data for statistical significance")
else:
    print(f" NEUTRAL. 0.00% return in 7 days")

print("="*80)



 LIVE PAPER TRADING - LAST 7 DAYS (YFINANCE)

 Fetching last 7 days of NIFTY BANK data from yfinance...
 Fetched 1500 candles from 2026-01-27 09:15:00+05:30 to 2026-01-30 15:29:00+05:30
   Ticker: ^NSEBANK

 Applying feature engineering to live data...
 Features applied: 28 features
   ML Prob range: 0.2736 to 0.6226
   Data points: 1421 candles

 LIVE PAPER TRADING CONFIGURATION (7 DAYS - HYBRID OPTIMAL v2):
   Entry Threshold: 0.5266
   Stop Loss: 1.50%
   1st Target: 0.40%
   2nd Target: 0.80%
   3rd Target: 1.50%
   Trailing Stop: 0.30%
   Holding Period: 20 bars
   Position Size: 15% - 50%

 Starting Live Paper Trading (1421 candles)...

🟢 ENTRY #1 | 2026-01-27 11:04:00 | ₹58503.80 | Conf: 54.8% | Size: 16% | Capital: ₹1,000,000
🟢 ENTRY #2 | 2026-01-27 11:05:00 | ₹58557.10 | Conf: 53.4% | Size: 15% | Capital: ₹1,000,000
🟢 ENTRY #3 | 2026-01-27 11:09:00 | ₹58524.95 | Conf: 53.9% | Size: 14% | Capital: ₹1,000,000
🟢 ENTRY #4 | 2026-01-27 11:15:00 | ₹58562.05 | Conf: 53.2% | Size: 14

In [13]:
import yfinance as yf
from datetime import datetime, timedelta
import pytz

IST = pytz.timezone("Asia/Kolkata")

print("\n" + "="*80)
print(" LIVE MARKET SIMULATION - TODAY'S TRADING ")
print("="*80)

print("\n Fetching today's NIFTY BANK intraday data from yfinance...")

try:
    trade_date = datetime.now(IST).date() - timedelta(days=4)
    start_time = IST.localize(datetime.combine(trade_date, datetime.min.time()))
    end_time   = IST.localize(datetime.combine(trade_date, datetime.max.time()))

    print(f"   Fetching from: {start_time} to {end_time}")

    today_data = yf.download(
        "^NSEBANK",
        start=start_time,
        end=end_time,
        interval="1m",
        progress=False
    )
    # --- FORCE YFINANCE DATA TO IST ---
    if today_data.index.tz is None:
        today_data.index = today_data.index.tz_localize("UTC").tz_convert(IST)
    else:
        today_data.index = today_data.index.tz_convert(IST)


    if isinstance(today_data.columns, pd.MultiIndex):
        today_data.columns = [col[0] for col in today_data.columns]

    today_data.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    today_data = today_data[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()

    print(f" Fetched {len(today_data)} candles (yesterday)")
    print(f"   Time Range: {today_data.index[0]} to {today_data.index[-1]}")
    print(f"   Price Range: ₹{today_data['Low'].min():.2f} - ₹{today_data['High'].max():.2f}")

except Exception as e:
    print(" Error fetching data:", e)
    print("   Using fallback: simulating with test data")
    
    today_data = test_df[['Open', 'High', 'Low', 'Close', 'Volume']].iloc[-390:].copy()
    today_data.index = pd.date_range(start=datetime.now() - timedelta(hours=6), periods=len(today_data), freq='1min')
    print(f" Using {len(today_data)} candles from test data (fallback)")

print("\n Applying feature engineering to today's data...")

try:
    today_with_signals = add_scalping_signals(today_data.copy())
    today_with_features = add_basic_features(today_with_signals.copy())
    today_features = add_advanced_features(today_with_features.copy())
    today_features = today_features.dropna()
    
    X_today = today_features[feature_cols].copy()
    X_today_scaled = scaler_bt.transform(X_today)
    ml_prob_today = model_bt.predict_proba(X_today_scaled)[:, 1]
    
    today_prices = today_features["Close"].values
    today_atr = today_features["atr"].values
    
    print(f" Features applied successfully")
    print(f"   ML Probability range: {ml_prob_today.min():.4f} to {ml_prob_today.max():.4f}")
    print(f"   Mean confidence: {ml_prob_today.mean():.4f}")
    print(f"   Data points: {len(today_features)} candles")
    
except Exception as e:
    print(f" Error in feature pipeline: {e}")
    raise


TODAY_ENTRY_THRESHOLD = np.percentile(ml_prob_today, 55)  
TODAY_STOP_LOSS = 0.0100  
TODAY_TAKE_PROFIT_1 = 0.0025  
TODAY_TAKE_PROFIT_2 = 0.0050  
TODAY_TAKE_PROFIT_3 = 0.0100  
TODAY_TRAIL_STOP = 0.0020  
TODAY_HORIZON = 15  
TODAY_MIN_POSITION = 0.05  
TODAY_MAX_POSITION = 0.15  
TODAY_COOLDOWN = 0


today_capital = INITIAL_CAPITAL
today_peak_capital = today_capital
today_positions = []
today_trades = []
today_equity_curve = []
today_last_exit_idx = -TODAY_COOLDOWN


for i in range(len(today_features)):
    current_price = today_prices[i]
    current_time = today_features.index[i]
    current_ml_prob = ml_prob_today[i]
    current_atr = today_atr[i]
    
    # ------ WARM-UP (First 20 candles) ------
    if i < 20:
        today_equity_curve.append(today_capital)
        continue
    
    # ------ UPDATE EXISTING POSITIONS (MULTIPLE) ------
    positions_to_close = []
    
    for pos_idx, pos in enumerate(today_positions):
        price_move = (current_price - pos['entry_price']) / pos['entry_price']
        
        # Volatility-adjusted stop loss
        vol_adjusted_stop = TODAY_STOP_LOSS * (1 + (current_atr / current_price) * 0.3)
        vol_adjusted_stop = np.clip(vol_adjusted_stop, TODAY_STOP_LOSS * 0.8, TODAY_STOP_LOSS * 1.2)
        
        exit_hit = False
        exit_reason = None
        exit_price = current_price
        
      
        # ================================================
        
        
        if price_move >= TODAY_TAKE_PROFIT_1:
            exit_hit = True
            exit_reason = "QUICK_PROFIT"
            exit_price = pos['entry_price'] * (1 + TODAY_TAKE_PROFIT_1)
        
        # Tier 2: Medium 0.50% exit (ACHIEVABLE)
        elif price_move >= TODAY_TAKE_PROFIT_2:
            exit_hit = True
            exit_reason = "MEDIUM_PROFIT"
            exit_price = pos['entry_price'] * (1 + TODAY_TAKE_PROFIT_2)
        
        # Tier 3: Big 1.00% exit (ambitious)
        elif price_move >= TODAY_TAKE_PROFIT_3:
            exit_hit = True
            exit_reason = "BIG_PROFIT"
            exit_price = pos['entry_price'] * (1 + TODAY_TAKE_PROFIT_3)
        
        # Trailing stop - Let winner run
        elif price_move > TODAY_TAKE_PROFIT_1 and current_price < pos['max_price'] * (1 - TODAY_TRAIL_STOP):
            exit_hit = True
            exit_reason = "TRAIL"
            exit_price = pos['max_price'] * (1 - TODAY_TRAIL_STOP)
        
        # Stop loss - Protect capital
        elif price_move <= -vol_adjusted_stop:
            exit_hit = True
            exit_reason = "STOP"
            exit_price = current_price
        
        # Time-based exit - Last resort
        elif i - pos['entry_idx'] >= TODAY_HORIZON:
            exit_hit = True
            exit_reason = "TIME"
            exit_price = current_price
        
        if exit_hit:
            # Calculate P&L
            ret = (exit_price - pos['entry_price']) / pos['entry_price']
            net_ret = ret - (COST_PER_TRADE * 2)
            pnl = pos['position_value'] * net_ret
            
            today_capital += pnl
            today_peak_capital = max(today_peak_capital, today_capital)
            
            # Log trade
            today_trades.append({
                'entry_idx': pos['entry_idx'],
                'exit_idx': i,
                'entry_price': pos['entry_price'],
                'exit_price': exit_price,
                'entry_time': pos['entry_time'],
                'exit_time': current_time,
                'prob': pos['ml_prob'],
                'size': pos['position_fraction'],
                'return': net_ret,
                'pnl': pnl,
                'capital_after': today_capital,
                'exit_reason': exit_reason
            })
            
            # Print exit
            emoji = "🎯" if pnl > 0 else "🛑"
            pnl_str = f"+₹{pnl:,.0f}" if pnl > 0 else f"-₹{abs(pnl):,.0f}"
            ret_str = f"+{net_ret*100:.2f}%" if net_ret > 0 else f"{net_ret*100:.2f}%"
            
            print(f"{emoji} EXIT | {current_time.strftime('%Y-%m-%d %H:%M:%S')} | ₹{exit_price:.2f} | P&L: {pnl_str} ({ret_str}) | [{exit_reason}] | Capital: ₹{today_capital:,.0f}")
            
            positions_to_close.append(pos_idx)
            today_last_exit_idx = i
    
    # Remove closed positions
    today_positions = [pos for idx, pos in enumerate(today_positions) if idx not in positions_to_close]
    
    # ------ SMART ENTRY LOGIC (BALANCE QUALITY + QUANTITY) ------
    if current_ml_prob >= TODAY_ENTRY_THRESHOLD and i < len(today_features) - TODAY_HORIZON:
        # Check momentum
        momentum_5 = today_features["momentum_5"].iloc[i]
        if momentum_5 >= TREND_STRENGTH_MIN * 0.5:  
            
            # Allow multiple positions with smart exposure management
            total_exposure = sum(p['position_fraction'] for p in today_positions)
            max_allowed = 1.0  # Max 100% total exposure
            if total_exposure < max_allowed:
                
                # Position sizing: Balance between confidence and exposure
                confidence_edge = current_ml_prob - TODAY_ENTRY_THRESHOLD
                max_edge = 1.0 - TODAY_ENTRY_THRESHOLD
                normalized = confidence_edge / max_edge if max_edge > 0 else 0.4
                
                # Hybrid sizing: quality + volume
                position_fraction = TODAY_MIN_POSITION + (TODAY_MAX_POSITION - TODAY_MIN_POSITION) * (normalized ** 1.3)
                position_fraction = np.clip(position_fraction, TODAY_MIN_POSITION, TODAY_MAX_POSITION)
                
                # Reduce if portfolio heavily exposed
                position_fraction *= (1.0 - total_exposure / max_allowed * 0.3)
                position_fraction = max(position_fraction, TODAY_MIN_POSITION * 0.5)
                
                position_value = today_capital * position_fraction
                
                # Open position
                today_positions.append({
                    'entry_idx': i,
                    'entry_price': current_price,
                    'entry_time': current_time,
                    'position_value': position_value,
                    'position_fraction': position_fraction,
                    'ml_prob': current_ml_prob,
                    'max_price': current_price
                })
                
                prob_pct = current_ml_prob * 100
                size_pct = position_fraction * 100
                print(f"🟢 ENTRY #{len(today_positions)} | {current_time.strftime('%Y-%m-%d %H:%M:%S')} | ₹{current_price:.2f} | Conf: {prob_pct:.1f}% | Size: {size_pct:.0f}% | Capital: ₹{today_capital:,.0f}")
    
    # Update max price for trailing stop
    for pos in today_positions:
        pos['max_price'] = max(pos['max_price'], current_price)
    
    today_equity_curve.append(today_capital)

# Force close any remaining open positions at market close
if today_positions:
    print(f"\n⏹️  MARKET CLOSE - Closing {len(today_positions)} open position(s)...")
    for pos in today_positions:
        exit_price = today_prices[-1]
        ret = (exit_price - pos['entry_price']) / pos['entry_price']
        net_ret = ret - (COST_PER_TRADE * 2)
        pnl = pos['position_value'] * net_ret
        
        today_capital += pnl
        today_peak_capital = max(today_peak_capital, today_capital)
        
        today_trades.append({
            'entry_idx': pos['entry_idx'],
            'exit_idx': len(today_features) - 1,
            'entry_price': pos['entry_price'],
            'exit_price': exit_price,
            'entry_time': pos['entry_time'],
            'exit_time': today_features.index[-1],
            'prob': pos['ml_prob'],
            'size': pos['position_fraction'],
            'return': net_ret,
            'pnl': pnl,
            'capital_after': today_capital,
            'exit_reason': 'CLOSE'
        })
        
        print(f"🛑 CLOSE | Final Price: ₹{exit_price:.2f} | P&L: ₹{pnl:,.0f} ({net_ret*100:+.2f}%)")

# ============= RESULTS ANALYSIS =============
print("\n" + "="*80)
#print("📊 TODAY'S HYBRID OPTIMAL v2+ ENHANCED - FINAL RESULTS")
print("="*80)

today_equity = np.array(today_equity_curve)
today_returns = np.diff(today_equity) / today_equity[:-1]

today_total_return = (today_capital - INITIAL_CAPITAL) / INITIAL_CAPITAL
today_total_pnl = today_capital - INITIAL_CAPITAL
today_max_dd = ((today_equity / np.maximum.accumulate(today_equity)) - 1).min()

if len(today_returns) > 0:
    today_sharpe = (today_returns.mean() / today_returns.std() if today_returns.std() > 0 else 0) * np.sqrt(252 * 6.5 * 60)
else:
    today_sharpe = 0

today_winning = sum(1 for t in today_trades if t['pnl'] > 0)
today_losing = len(today_trades) - today_winning
today_win_rate = today_winning / len(today_trades) if len(today_trades) > 0 else 0

today_gross_profit = sum(t['pnl'] for t in today_trades if t['pnl'] > 0)
today_gross_loss = abs(sum(t['pnl'] for t in today_trades if t['pnl'] < 0))
today_pf = today_gross_profit / today_gross_loss if today_gross_loss > 0 else (np.inf if today_gross_profit > 0 else 0)

today_avg_win = today_gross_profit / today_winning if today_winning > 0 else 0
today_avg_loss = today_gross_loss / today_losing if today_losing > 0 else 0

print(f"\n CAPITAL SUMMARY:")
print(f"   Starting Capital:   ₹{INITIAL_CAPITAL:,.0f}")
print(f"   Ending Capital:     ₹{today_capital:,.0f}")
print(f"   Peak Capital:       ₹{today_peak_capital:,.0f}")
print(f"   Net P&L:            ₹{today_total_pnl:+,.0f}")

print(f"\n PERFORMANCE METRICS:")
print(f"   Daily Return:       {today_total_return*100:+.3f}%")
print(f"   Max Drawdown:       {today_max_dd*100:.2f}%")
print(f"   Sharpe Ratio:       {today_sharpe:.2f}")
print(f"   Profit Factor:      {today_pf:.2f}" if today_pf != np.inf else f"   Profit Factor:      ∞ (All wins)")



# Exit reason breakdown
exit_reasons = {}
for t in today_trades:
    reason = t['exit_reason']
    exit_reasons[reason] = exit_reasons.get(reason, 0) + 1

print(f"\n EXIT DISTRIBUTION:")
total_trades_count = len(today_trades) if len(today_trades) > 0 else 1
for reason in sorted(exit_reasons.keys()):
    count = exit_reasons[reason]
    print(f"   {reason:15} : {count:3} trades ({count/total_trades_count*100:5.1f}%)")


# Final verdict for today

print("="*80)

if len(today_trades) == 0:
    print("⚪ NO TRADES TODAY")
   
    
elif today_total_return > 0.25:
    print(f" EXCEPTIONAL DAY! +{today_total_return*100:.3f}% return")
    print(f"   {len(today_trades)} quality trades")
    print(f"   Win Rate: {today_win_rate*100:.1f}% | Profit Factor: {today_pf:.2f}")
    
    
elif today_total_return > 0.10:
    print(f" EXCELLENT DAY! +{today_total_return*100:.3f}% return")
    print(f"   {len(today_trades)} trades with strong execution")
    print(f"   Win Rate: {today_win_rate*100:.1f}% | Profit Factor: {today_pf:.2f}")
    
elif today_total_return > 0.05:
    print(f" VERY GOOD DAY! +{today_total_return*100:.3f}% return")
    print(f"   {len(today_trades)} trades with {today_win_rate*100:.1f}% win rate")
    
elif today_total_return > 0:
    print(f" PROFITABLE DAY! +{today_total_return*100:.3f}% return")
    print(f"   {len(today_trades)} trades executed successfully")
    
elif today_total_return == 0:
    print(" BREAKEVEN TODAY")
    
else:
    print(f"  LOSS TODAY: {today_total_return*100:.3f}%")
    

print("="*80 + "\n")


 LIVE MARKET SIMULATION - TODAY'S TRADING 

 Fetching today's NIFTY BANK intraday data from yfinance...
   Fetching from: 2026-01-27 00:00:00+05:30 to 2026-01-27 23:59:59.999999+05:30
 Fetched 375 candles (yesterday)
   Time Range: 2026-01-27 09:15:00+05:30 to 2026-01-27 15:29:00+05:30
   Price Range: ₹58124.35 - ₹59433.80

 Applying feature engineering to today's data...
 Features applied successfully
   ML Probability range: 0.4506 to 0.6177
   Mean confidence: 0.5211
   Data points: 296 candles
🟢 ENTRY #1 | 2026-01-27 10:58:00 | ₹58469.90 | Conf: 53.3% | Size: 5% | Capital: ₹1,000,000
🟢 ENTRY #2 | 2026-01-27 10:59:00 | ₹58450.15 | Conf: 53.4% | Size: 5% | Capital: ₹1,000,000
🟢 ENTRY #3 | 2026-01-27 11:00:00 | ₹58429.25 | Conf: 55.8% | Size: 5% | Capital: ₹1,000,000
🟢 ENTRY #4 | 2026-01-27 11:01:00 | ₹58447.50 | Conf: 52.8% | Size: 5% | Capital: ₹1,000,000
🟢 ENTRY #5 | 2026-01-27 11:02:00 | ₹58460.65 | Conf: 54.1% | Size: 5% | Capital: ₹1,000,000
🟢 ENTRY #6 | 2026-01-27 11:04:00 | ₹

# 🔴 SUMMARY: Live Paper Trading Features Added to Streamlit App

## New Tab: "Real-Time Market (9:15 AM - 3:15 PM)"

The Streamlit app (`app.py`) now includes a **5th tab** for live paper trading that:

✅ **Real-Time Market Integration**
- Fetches live 1-minute NIFTY BANK data from yfinance
- Synchronized with NSE market hours (9:15 AM - 3:15 PM IST)
- Live market status indicator

✅ **Advanced Trading Configuration**
- Entry threshold (percentile-based)
- Dynamic stop loss & take profit
- Trailing stops for winner management
- Position sizing controls
- Max open positions limit

✅ **Multi-Position Management**
- Support for 1-5 simultaneous positions
- Intelligent exposure management
- Adaptive position sizing based on confidence

✅ **Real-Time Results**
- Live equity curve visualization
- Trade-by-trade P&L breakdown
- Exit reason distribution
- Performance metrics (Sharpe, Max DD, Profit Factor)
- Win rate and trade statistics

## How to Access:
1. Run Streamlit: `streamlit run app.py`
2. Navigate to the **"🔴 Real-Time Market (9:15 AM - 3:15 PM)"** tab
3. Configure parameters and click **"▶️ START REAL-TIME TRADING SIMULATION"**
4. View live trades, capital evolution, and detailed analytics

## Features:
- 🎯 Tiered profit targets (quick, medium, big)
- 🛑 Volatility-adjusted stops
- 📊 Equity curve tracking
- 💹 Detailed trade analysis
- ⏰ Market hours awareness
- 🔔 Real-time status indicators
